<a href="https://colab.research.google.com/github/Rajarajan1087/fde-learning-lab/blob/FinanceFlow/FDE%20Projects/FinanceFlow/FinFlow_Great_Expectations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏪 FinFlow — Great Expectations Demo
## **FDE Engineering Track  ·**

### FinanceFlow's Credit Risk policy requires that every loan record entering the model satisfies a formal data contract. Implement these six expectations as a GX checkpoint. Above each one, write a Python comment explaining the underwriting rule it encodes.
---

## Cell 1 — Install

In [2]:
!pip install great-expectations --quiet
print("✅ great-expectations installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.4 MB/s eta 0:00:00
✅ great-expectations installed


## Cell 2 — Imports

In [3]:
import great_expectations as gx
import pandas as pd
import os
from IPython.display import display, HTML

print(f"✅ great-expectations {gx.__version__}")

✅ great-expectations 1.18.0


## Cell 3 — Load the RetailCo dataset

The dataset is pre-loaded. This is the CRM export RetailCo handed us on Day 1.

In [5]:
# RetailCo dataset — pre-loaded for the demo
import io



df = pd.read_csv('financeflow_raw.csv')

print("✅ FinFlow dataset loaded")
print(f"   Rows    : {len(df):,}")
print(f"   Columns : {list(df.columns)}")
print()
df.head(8)

✅ FinFlow dataset loaded
   Rows    : 400
   Columns : ['loan_id', 'business_age_years', 'sector', 'num_employees', 'annual_revenue_usd', 'loan_amount_usd', 'loan_term_months', 'interest_rate_pct', 'credit_score', 'debt_to_income_ratio', 'loan_purpose', 'state', 'default_flag']



,loan_id,business_age_years,sector,num_employees,annual_revenue_usd,loan_amount_usd,loan_term_months,interest_rate_pct,credit_score,debt_to_income_ratio,loan_purpose,state,default_flag
0,L00001,21,Healthcare,357,4599846.93,466040.52,24,11.99,770.0,0.259,Equipment,WA,0.0
1,L00002,20,Retail,445,1827243.90,14695.89,24,7.13,716.0,0.630,Refinancing,OH,0.0
2,L00003,16,Hospitality,365,4207464.42,249462.76,12,12.13,632.0,0.261,Real Estate,FL,0.0
3,L00004,6,Agriculture,236,2849969.78,388548.34,48,11.92,577.0,0.415,Real Estate,NC,0.0
4,L00005,12,Healthcare,423,3796091.90,382402.02,48,11.88,737.0,0.156,Equipment,IL,0.0
5,L00006,2,Hospitality,136,1775456.04,410197.01,48,16.18,580.0,0.527,Equipment,FL,1.0
6,L00007,24,Healthcare,354,1785622.66,358269.44,48,6.74,591.0,0.532,Equipment,CA,1.0
7,L00008,19,Technology,274,1441370.84,273996.59,36,17.72,669.0,0.259,Real Estate,NC,0.0


## Cell 4 — Create GX context

In [6]:
context = gx.get_context(mode="file")
print("GX context ready")

GX context ready


## Cell 5 - Register dataset + build pre-built Expectation Suite

These are the 6 rules already written for the Financial Flow SMB data.  

1.	credit_score must not be null.
2.	credit_score must be between 300 and 850 (valid FICO range).
3.	default_flag must not be null.
4.	annual_revenue_usd must be greater than or equal to 0.
5.	debt_to_income_ratio must be between 0.05 and 1.00.
6.	sector must be one of: Retail, Manufacturing, Technology, Hospitality, Healthcare, Construction, Agriculture.

In [ ]:
# --- Data asset ---
data_source      = context.data_sources.add_pandas("FinFlow_datasource")
data_asset       = data_source.add_dataframe_asset(name="FinFlow_LoanApp")
batch_definition = data_asset.add_batch_definition_whole_dataframe("full_batch")

# --- Pre-built Expectation Suite ---
suite = context.suites.add(gx.ExpectationSuite(name="FinFlow_suite"))

# credit_score: expect not null
# This is the target variable. A churn model cannot be trained
# if 31% of the labels are missing.
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="credit_score")
)

# credit_score: expect values between 300 and 850
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="credit_score",
        min_value=300,
        max_value=850
    )
)



# default_flag : expect not null
# This is the target variable. A churn model cannot be trained
# if 31% of the labels are missing.
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="default_flag")
)

# annual_revenue_usd: expect values >= 0
suite.add_expectation(
    gx.expectations.ExpectColumnValuesTobeNonNegative(column="annual_revenue_usd")
)

#debt_to_income_ratio must be between 0.05 and 1.00.
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="debt_to_income_ratio",
        min_value=0.05,
        max_value=1.00
    )
)

# sector must be one of: Retail, Manufacturing, Technology, Hospitality, Healthcare, Construction, Agriculture.
suite.add_expectation(
    gx.expectations.ExpectColumnValueTobeOneOf(
        column="sector",
        value_set=["Retail", "Manufacturing", "Technology", "Hospitality", "Healthcare", "Construction", "Agriculture"]
    )
)

print("✅ Pre-built checkpoint ready — 6 expectations loaded")

✅ Pre-built checkpoint ready — 5 expectations loaded


## Cell 6 — Run the checkpoint

> **Trainer:** run this cell live. The results will show which rules pass and which fail.

In [ ]:
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="retailco_validation",
        data=batch_definition,
        suite=suite,
    )
)

checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name="retailco_checkpoint",
        validation_definitions=[validation_definition],
    )
)

results = checkpoint.run(batch_parameters={"dataframe": df})

print(results)

Calculating Metrics:   0%|          | 0/30 [00:00<?, ?it/s]

run_id={"run_name": null, "run_time": "2026-06-06T07:58:43.774055+00:00"} run_results={ValidationResultIdentifier::retailco_suite/__none__/20260606T075843.774055Z/retailco_datasource-retailco_sales: {
  "success": false,
  "results": [
    {
      "success": false,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "retailco_datasource-retailco_sales",
          "column": "customer_id"
        },
        "meta": {},
        "id": "0db4203a-0a66-4927-b289-ba9f440c4482",
        "severity": "critical"
      },
      "result": {
        "element_count": 89,
        "unexpected_count": 30,
        "unexpected_percent": 33.70786516853933,
        "partial_unexpected_list": [
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
     

## Cell 7 — Read the Data Docs report

> *"This report is not for you — it is for the client.*  
> *You email this to the Head of Sales before the 10 AM meeting.*  
> *It is evidence, not opinion."*

**Learners:** read the report below. Two checks fail, three pass.  
Which two failures directly block the AI use case — and why?

In [ ]:
context.build_data_docs()

# Find and display the validation report inline
docs_root = "gx/uncommitted/data_docs/local_site/validations"
html_file = None

for root, dirs, filenames in os.walk(docs_root):
    for f in filenames:
        if f.endswith(".html"):
            html_file = os.path.join(root, f)

if html_file:
    with open(html_file, "r") as f:
        html_content = f.read()
    print(f"📄 Showing: {html_file}\n")
    display(HTML(html_content))
else:
    print("⚠️  Data Docs not found. Make sure Cell 6 ran successfully.")

📄 Showing: gx/uncommitted/data_docs/local_site/validations/retailco_suite/__none__/20260606T075843.774055Z/retailco_datasource-retailco_sales.html



,
Evaluated Expectations,5
Successful Expectations,2
Unsuccessful Expectations,3
Success Percent,40%
,
Great Expectations Version,1.18.0
Run Name,__none__
Run Time,2026-06-06T07:58:43Z
,
ge_load_time,20260606T075843.786424Z


## Cell 8 — Download the Data Docs report

Download the report to share with the client before the 10 AM meeting.

In [ ]:
from google.colab import files

docs_root = "gx/uncommitted/data_docs/local_site/validations"
html_file = None

for root, dirs, filenames in os.walk(docs_root):
    for f in filenames:
        if f.endswith(".html"):
            html_file = os.path.join(root, f)

if html_file:
    files.download(html_file)
    print("✅ Data Docs report downloaded — send this to the client before 10 AM")
else:
    print("⚠️  No report found. Run Cell 7 first.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Data Docs report downloaded — send this to the client before 10 AM



---
## 📝 Learner Discussion Questions

After reading the Data Docs report, discuss with the class:

**1.** Two critical checks fail — customer identity and churn label.  
In your own words, why does each one block the AI use case?

**2.** Three checks pass. What does that mean for the project — is it cancelled?

**3.** Write one sentence for the CFO explaining the `customer_id` finding.  
Rules: no technical terms, no percentages, must include a business consequence.

> ❌ *"44% null rate on customer_id"*  
> ✅ *"We cannot attribute nearly half of your sales transactions to a specific customer"*
